# Kalenjin ASR Data Processing v2

Complete processing pipeline for Mozilla Common Voice Kalenjin dataset.

## Pipeline Overview
1. Data Loading & Validation
2. Audio Processing & Feature Extraction
3. Text Preprocessing
4. Dataset Preparation
5. Export for Training

In [ ]:
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
from pathlib import Path
import json
import re
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuration
SAMPLE_RATE = 16000
MAX_DURATION = 10.0
MIN_DURATION = 1.0

print("✓ Libraries loaded")

## 1. Data Loading & Validation

In [ ]:
# Load datasets
DATA_PATH = Path('../cv-corpus-24.0-2025-12-05-kln/cv-corpus-24.0-2025-12-05/kln')
CLIPS_PATH = DATA_PATH / 'clips'
OUTPUT_PATH = Path('../processed_data')
OUTPUT_PATH.mkdir(exist_ok=True)

# Load splits
train_df = pd.read_csv(DATA_PATH / 'train.tsv', sep='\t')
dev_df = pd.read_csv(DATA_PATH / 'dev.tsv', sep='\t')
test_df = pd.read_csv(DATA_PATH / 'test.tsv', sep='\t')

print(f"Train: {len(train_df)} samples")
print(f"Dev: {len(dev_df)} samples")
print(f"Test: {len(test_df)} samples")

## 2. Audio Processing Functions

In [ ]:
def load_and_preprocess_audio(audio_path):
    """Load and preprocess audio file"""
    try:
        audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
        
        # Remove silence
        audio, _ = librosa.effects.trim(audio, top_db=20)
        
        # Normalize
        audio = librosa.util.normalize(audio)
        
        duration = len(audio) / SAMPLE_RATE
        
        return audio, duration
    except Exception as e:
        return None, 0

def extract_features(audio):
    """Extract mel-spectrogram features"""
    mel_spec = librosa.feature.melspectrogram(
        y=audio, 
        sr=SAMPLE_RATE,
        n_mels=80,
        hop_length=160,
        win_length=400
    )
    return librosa.power_to_db(mel_spec)

print("✓ Audio processing functions defined")

## 3. Text Preprocessing

In [ ]:
def clean_text(text):
    """Clean and normalize text"""
    if pd.isna(text):
        return ""
    
    # Convert to lowercase
    text = text.lower().strip()
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    
    # Remove punctuation except apostrophes
    text = re.sub(r"[^\w\s']", '', text)
    
    return text

def build_vocabulary(texts):
    """Build character vocabulary"""
    chars = set()
    for text in texts:
        chars.update(text)
    
    vocab = ['<pad>', '<unk>', '<sos>', '<eos>'] + sorted(list(chars))
    char_to_idx = {char: idx for idx, char in enumerate(vocab)}
    idx_to_char = {idx: char for idx, char in enumerate(vocab)}
    
    return vocab, char_to_idx, idx_to_char

print("✓ Text processing functions defined")

## 4. Process Training Data

In [ ]:
def process_split(df, split_name):
    """Process a data split"""
    processed_data = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        audio_path = CLIPS_PATH / row['path']
        
        if not audio_path.exists():
            continue
            
        # Load audio
        audio, duration = load_and_preprocess_audio(audio_path)
        
        if audio is None or duration < MIN_DURATION or duration > MAX_DURATION:
            continue
            
        # Clean text
        text = clean_text(row['sentence'])
        
        if not text:
            continue
            
        # Extract features
        features = extract_features(audio)
        
        processed_data.append({
            'audio_path': str(audio_path),
            'text': text,
            'duration': duration,
            'features_shape': features.shape,
            'client_id': row.get('client_id', ''),
            'age': row.get('age', ''),
            'gender': row.get('gender', '')
        })
    
    return processed_data

# Process all splits
train_processed = process_split(train_df, 'train')
dev_processed = process_split(dev_df, 'dev')
test_processed = process_split(test_df, 'test')

print(f"Processed - Train: {len(train_processed)}, Dev: {len(dev_processed)}, Test: {len(test_processed)}")

## 5. Build Vocabulary

In [ ]:
# Collect all texts
all_texts = [item['text'] for item in train_processed + dev_processed + test_processed]

# Build vocabulary
vocab, char_to_idx, idx_to_char = build_vocabulary(all_texts)

print(f"Vocabulary size: {len(vocab)}")
print(f"Characters: {vocab[4:]}")

# Save vocabulary
vocab_data = {
    'vocab': vocab,
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char
}

with open(OUTPUT_PATH / 'vocabulary.json', 'w', encoding='utf-8') as f:
    json.dump(vocab_data, f, ensure_ascii=False, indent=2)

print("✓ Vocabulary saved")

## 6. Dataset Statistics

In [ ]:
def compute_stats(processed_data, split_name):
    """Compute dataset statistics"""
    durations = [item['duration'] for item in processed_data]
    text_lengths = [len(item['text']) for item in processed_data]
    
    stats = {
        'split': split_name,
        'samples': len(processed_data),
        'total_duration': sum(durations),
        'avg_duration': np.mean(durations),
        'min_duration': min(durations),
        'max_duration': max(durations),
        'avg_text_length': np.mean(text_lengths),
        'min_text_length': min(text_lengths),
        'max_text_length': max(text_lengths)
    }
    
    return stats

# Compute statistics
train_stats = compute_stats(train_processed, 'train')
dev_stats = compute_stats(dev_processed, 'dev')
test_stats = compute_stats(test_processed, 'test')

all_stats = [train_stats, dev_stats, test_stats]

# Display statistics
stats_df = pd.DataFrame(all_stats)
print("\nDataset Statistics:")
print(stats_df.round(2))

# Save statistics
stats_df.to_csv(OUTPUT_PATH / 'dataset_stats.csv', index=False)
print("\n✓ Statistics saved")

## 7. Save Processed Data

In [ ]:
# Save processed datasets
datasets = {
    'train': train_processed,
    'dev': dev_processed,
    'test': test_processed
}

for split_name, data in datasets.items():
    # Save as JSON
    with open(OUTPUT_PATH / f'{split_name}_processed.json', 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    
    # Save as CSV for easy inspection
    df = pd.DataFrame(data)
    df.to_csv(OUTPUT_PATH / f'{split_name}_processed.csv', index=False)
    
    print(f"✓ {split_name} data saved ({len(data)} samples)")

print(f"\n✓ All processed data saved to {OUTPUT_PATH}")

## 8. Create Training Configuration

In [ ]:
# Create training configuration
config = {
    'dataset': {
        'name': 'kalenjin_asr',
        'language': 'kln',
        'sample_rate': SAMPLE_RATE,
        'vocab_size': len(vocab),
        'splits': {
            'train': len(train_processed),
            'dev': len(dev_processed),
            'test': len(test_processed)
        }
    },
    'audio': {
        'sample_rate': SAMPLE_RATE,
        'n_mels': 80,
        'hop_length': 160,
        'win_length': 400,
        'max_duration': MAX_DURATION,
        'min_duration': MIN_DURATION
    },
    'text': {
        'vocab_size': len(vocab),
        'pad_token': '<pad>',
        'unk_token': '<unk>',
        'sos_token': '<sos>',
        'eos_token': '<eos>'
    },
    'paths': {
        'data_dir': str(OUTPUT_PATH),
        'vocab_file': 'vocabulary.json',
        'train_file': 'train_processed.json',
        'dev_file': 'dev_processed.json',
        'test_file': 'test_processed.json'
    }
}

# Save configuration
with open(OUTPUT_PATH / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("✓ Training configuration saved")
print(f"\nProcessing complete! Files saved to: {OUTPUT_PATH}")
print(f"Total processed samples: {sum(len(data) for data in datasets.values())}")

## Summary

This notebook has successfully:

1. **Loaded** Mozilla Common Voice Kalenjin dataset
2. **Processed** audio files (trimming, normalization, feature extraction)
3. **Cleaned** text data (normalization, punctuation removal)
4. **Built** character-level vocabulary
5. **Generated** dataset statistics
6. **Exported** processed data in multiple formats
7. **Created** training configuration file

The processed data is ready for ASR model training with frameworks like Wav2Vec2, Whisper, or custom architectures.